# Hybrid quantum-classical maize yield forecasting

This notebook builds a county-year modeling table for Kenya, keeps rainfall/weather features separate from contextual features, uses a leakage-safe chronological split, and fits a quantum-kernel residual model. The quantum model is intentionally small (4 contextual features) so it is practical on a local simulator.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

DATA = Path('/home/jovyan/qulima_maize_yield.csv')
OUT = DATA.parent / 'maize_quantum_outputs'; OUT.mkdir(exist_ok=True)
df = pd.read_csv(DATA)
print('shape:', df.shape)
df.head()

In [ ]:
print('Columns and dtypes:')
print(df.dtypes.to_string())
print('\nCounty count:', df.county.nunique(), sorted(df.county.unique()))
print('Years:', df.year.min(), 'to', df.year.max(), 'unique:', df.year.nunique())
print('Season basis:', df.season_basis.value_counts(dropna=False).to_dict())
missing = df.isna().sum()
print('Missing values:', missing[missing.gt(0)].to_dict())
print(df.describe(include='all').T[['count','mean','min','max']].to_string())

## Modeling table

The source contains annual observations (rather than separate seasonal rows). The `season_basis` field is retained; when March–June observations are supplied, the same code will filter to them. Rainfall/weather variables are explicitly separated from non-rainfall contextual variables. Source URL columns and post-harvest production are excluded to avoid target leakage.

In [ ]:
# Prefer March-June seasonal rows if present; otherwise use the annual county-year observations supplied.
season_mask = df['season_basis'].astype(str).str.lower().str.contains('mar|march|m-j|mj|season')
if season_mask.any():
    model = df.loc[season_mask].copy()
    season_note = 'Filtered to March-June/season rows'
else:
    model = df.copy()
    season_note = 'No March-June rows found; using annual county-year rows'
model = model.sort_values(['year','county']).reset_index(drop=True)
RAIN = ['rain_dekads_available','rain_total_jan_jun_mm','rain_lta_jan_jun_mm','rain_anomaly_pct','rain_mean_dekad_mm','rain_std_dekad_mm','rain_cv','wet_dekads_ge20mm','heavy_dekads_ge50mm','dry_dekads_lt10mm','longest_dry_spell_dekads','rain_onset_doy']
CONTEXT = ['county_lat','county_lon','maize_area_ha']
TARGET='yield_t_ha'
keep=['county','county_pcode','year','season_basis',TARGET]+RAIN+CONTEXT
model_table=model[keep].copy()
print(season_note); print(model_table.shape); display(model_table.head())
model_table.to_csv(OUT/'county_year_modeling_table.csv',index=False)

## Leakage-safe temporal validation

The final test is the latest 20% of years. Training-side medians and all fitted models are learned only from the earlier period. A rainfall-only Ridge is the operational baseline; contextual classical and quantum models predict residuals relative to it.

In [ ]:
years=np.sort(model_table.year.unique()); cutoff=years[max(1,int(np.ceil(len(years)*.8))-1)]
train=model_table[model_table.year<=cutoff].copy(); test=model_table[model_table.year>cutoff].copy()
if test.empty: # robust fallback for very short data
    cutoff=years[-2]; train=model_table[model_table.year<=cutoff].copy(); test=model_table[model_table.year>cutoff].copy()
print('train years',train.year.min(),train.year.max(),'test years',test.year.min(),test.year.max(),'rows',len(train),len(test))

def fit_predict(cols, estimator=Ridge(alpha=1.0)):
    pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',estimator)])
    pipe.fit(train[cols],train[TARGET]); return pipe, pipe.predict(test[cols])
rain_pipe,rain_pred=fit_predict(RAIN)
train_rain_pred=rain_pipe.predict(train[RAIN]); train_resid=train[TARGET].to_numpy()-train_rain_pred
print('Rainfall baseline RMSE/MAE/R2:', mean_squared_error(test[TARGET],rain_pred)**.5, mean_absolute_error(test[TARGET],rain_pred), r2_score(test[TARGET],rain_pred))

In [ ]:
# Classical comparators and residual contextual model
ctx=['county_lat','county_lon','maize_area_ha']
ctx_pipe,ctx_pred=fit_predict(ctx,RandomForestRegressor(n_estimators=250,min_samples_leaf=4,random_state=7,n_jobs=-1))
all_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',Ridge(alpha=10))])
all_pipe.fit(train[RAIN+ctx],train[TARGET]); all_pred=all_pipe.predict(test[RAIN+ctx])
models={'rainfall_ridge':rain_pred,'context_rf':ctx_pred,'all_features_ridge':all_pred}
for n,p in models.items(): print(n, 'RMSE %.4f MAE %.4f R2 %.4f'%(mean_squared_error(test[TARGET],p)**.5,mean_absolute_error(test[TARGET],p),r2_score(test[TARGET],p)))

## Quantum residual model

We use four contextual features (`county_lat`, `county_lon`, `maize_area_ha`, and a deterministic county code) and map each to `[0, π]`. The `FidelityQuantumKernel` computes a train/test Gram matrix. Kernel Ridge regression is then fit to the rainfall-baseline residuals. This is a quantum-kernel residual learner, not a claim that the quantum model is universally superior.

In [ ]:
# Small contextual subset (4 features), with county identity represented numerically and fitted only from train.
all_counties=sorted(model_table.county.astype(str).unique()); county_code={c:i/(max(1,len(all_counties)-1)) for i,c in enumerate(all_counties)}
for x in (train,test): x['county_code']=x.county.astype(str).map(county_code)
Q=['county_lat','county_lon','maize_area_ha','county_code']
qscale=Pipeline([('impute',SimpleImputer(strategy='median')),('angle',MinMaxScaler(feature_range=(0,np.pi)))])
Xtr=qscale.fit_transform(train[Q]); Xte=qscale.transform(test[Q])
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
feature_map=ZZFeatureMap(feature_dimension=len(Q),reps=2,entanglement='linear')
qkernel=FidelityQuantumKernel(feature_map=feature_map)
Ktr=qkernel.evaluate(Xtr); Kte=qkernel.evaluate(Xte,Xtr)
from sklearn.kernel_ridge import KernelRidge
qmodel=KernelRidge(alpha=1.0,kernel='precomputed').fit(Ktr,train_resid)
q_resid=qmodel.predict(Kte); quantum_pred=rain_pred+q_resid
models['quantum_residual']=quantum_pred
print('quantum kernel shape:',Ktr.shape)
for n,p in models.items(): print(n, 'RMSE %.4f MAE %.4f R2 %.4f'%(mean_squared_error(test[TARGET],p)**.5,mean_absolute_error(test[TARGET],p),r2_score(test[TARGET],p)))

In [ ]:
metrics=[]
for n,p in models.items(): metrics.append({'model':n,'RMSE':mean_squared_error(test[TARGET],p)**.5,'MAE':mean_absolute_error(test[TARGET],p),'R2':r2_score(test[TARGET],p)})
metrics=pd.DataFrame(metrics).sort_values('RMSE'); display(metrics); metrics.to_csv(OUT/'model_metrics.csv',index=False)
pred_out=test[['county','year',TARGET]].copy()
for n,p in models.items(): pred_out[n+'_prediction']=p
pred_out.to_csv(OUT/'test_predictions.csv',index=False)
fig,ax=plt.subplots(1,2,figsize=(14,5))
metrics.set_index('model')['RMSE'].plot.bar(ax=ax[0],title='Test RMSE (lower is better)',color='steelblue'); ax[0].tick_params(axis='x',rotation=30)
ax[1].scatter(test[TARGET],quantum_pred,alpha=.65,label='quantum residual'); lo,hi=test[TARGET].min(),test[TARGET].max(); ax[1].plot([lo,hi],[lo,hi],'k--'); ax[1].set(xlabel='Actual yield (t/ha)',ylabel='Predicted yield (t/ha)',title='Quantum residual predictions'); ax[1].legend()
plt.tight_layout(); fig.savefig(OUT/'model_comparison.png',dpi=160); display(fig)
print('Saved outputs to',OUT)